# Notebook 6: Complete DeepSeek Block

## Learning Objectives
- Assemble complete transformer block
- Understand Pre-LN (Pre-LayerNorm) architecture
- Combine MLA + MoE + DSA
- Build multi-layer stack

## Task 1: RMSNorm

### HINT: Root Mean Square Layer Norm
```python
def rms_norm(x, eps=1e-6):
    rms = torch.sqrt(torch.mean(x², dim=-1, keepdim=True) + eps)
    return x / rms * weight
```

Simpler than LayerNorm, works well for LLMs!

In [ ]:
import torch
import torch.nn as nn
import math

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_size))
        
    def forward(self, x):
        # x: [..., hidden]
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

# Test RMSNorm
norm = RMSNorm(512)
x = torch.randn(2, 8, 512)
output = norm(x)
print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"Mean per token: {output.mean(-1)[0]}")

## Task 2: Complete DeepSeek Block

### HINT: Pre-LN Architecture
```
DeepSeek Block:
  x → RMSNorm → Attention → Add → RMSNorm → MoE → Add → output
```

This is **Pre-LN**: normalize BEFORE the sub-layer (more stable for deep networks)

In [ ]:
# Import from previous notebooks
from notebook_03_mla import MultiHeadLatentAttention
from notebook_04_moe import AuxiliaryLossFreeMoE

class DeepSeekBlock(nn.Module):
    """Complete DeepSeek transformer block"""
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config['hidden_size']
        
        # Pre-LN: Normalize BEFORE sub-layers
        self.attn_norm = RMSNorm(self.hidden_size)
        self.ffn_norm = RMSNorm(self.hidden_size)
        
        # Attention
        self.attention = MultiHeadLatentAttention(
            hidden_size=self.hidden_size,
            num_heads=config['num_heads'],
            num_kv_heads=config.get('num_kv_heads', 1)
        )
        
        # MoE
        self.moe = AuxiliaryLossFreeMoE(
            hidden_size=self.hidden_size,
            num_experts=config['num_experts'],
            num_active_experts=config['num_active_experts'],
            intermediate_size=config['intermediate_size']
        )
        
    def forward(self, x):
        # Pre-LN architecture
        # Attention block with residual
        attn_input = self.attn_norm(x)
        attn_output = self.attention(attn_input)
        x = x + attn_output  # Residual connection
        
        # MoE block with residual
        moe_input = self.ffn_norm(x)
        moe_output = self.moe(moe_input)
        x = x + moe_output  # Residual connection
        
        return x

# Test block
config = {
    'hidden_size': 512,
    'num_heads': 8,
    'num_kv_heads': 2,
    'num_experts': 8,
    'num_active_experts': 2,
    'intermediate_size': 1376
}

block = DeepSeekBlock(config)
x = torch.randn(2, 16, 512)
output = block(x)
print(f"Block input: {x.shape}")
print(f"Block output: {output.shape}")

## Task 3: Full Model Stack

### HINT: Stacking Blocks
DeepSeek V3.2 has 61 layers - we'll use 4 for our simplified version

In [ ]:
class DeepSeekV3(nn.Module):
    """Complete DeepSeek V3 model"""
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        # Token embeddings
        self.embeddings = nn.Embedding(config['vocab_size'], config['hidden_size'])
        
        # Stack of blocks
        self.layers = nn.ModuleList([
            DeepSeekBlock(config) for _ in range(config['num_layers'])
        ])
        
        # Final norm
        self.final_norm = RMSNorm(config['hidden_size'])
        
        # LM head (untied or tied with embeddings)
        self.lm_head = nn.Linear(config['hidden_size'], config['vocab_size'], bias=False)
        
    def forward(self, input_ids, return_logits=True):
        # Embeddings
        hidden_states = self.embeddings(input_ids)
        
        # Pass through layers
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        # Final norm
        hidden_states = self.final_norm(hidden_states)
        
        # LM head
        logits = self.lm_head(hidden_states)
        
        if return_logits:
            return logits
        return hidden_states

# Full config
config = {
    'vocab_size': 50000,
    'hidden_size': 512,
    'num_layers': 4,
    'num_heads': 8,
    'num_kv_heads': 2,
    'num_experts': 8,
    'num_active_experts': 2,
    'intermediate_size': 1376
}

model = DeepSeekV3(config)

# Count parameters
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print(f"Total parameters: {count_params(model)/1e6:.2f}M")

# Test forward pass
input_ids = torch.randint(0, config['vocab_size'], (2, 16))
logits = model(input_ids)

print(f"Input: {input_ids.shape}")
print(f"Logits: {logits.shape}")  # [batch, seq, vocab]

## Verification

Complete these checks:
1. ✅ Can implement RMSNorm
2. ✅ Can assemble complete DeepSeek block
3. ✅ Understand Pre-LN architecture
4. ✅ Can stack blocks into full model